In [ ]:
# Install required libraries for deep learning, data processing, and visualization
# TensorFlow → for building CNN and regression model
# Pandas → for handling tabular dataset
# NumPy → for numerical operations
# Scikit-learn → for preprocessing and evaluation
# Matplotlib & Seaborn → for visualization
!pip install tensorflow pandas numpy scikit-learn matplotlib seaborn

# Import basic libraries
import numpy as np
import pandas as pd
import os
import cv2
import matplotlib.pyplot as plt


# Import machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Import TensorFlow and Keras for building deep learning models
import tensorflow as tf
from tensorflow.keras import layers, models

# Import pretrained ResNet50 model for image feature extraction
from tensorflow.keras.applications import ResNet50

# Preprocessing function required before passing images to ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
# Import file upload utility for Google Colab
from google.colab import files

# Upload dataset files (housing.csv and image files)
uploaded = files.upload()

# Read the housing dataset CSV file into a pandas DataFrame
df = pd.read_csv("housing.csv")

# displaying the first 5 rows
df.head()

Saving housing.csv to housing.csv


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
# Drop missing values
df = df.dropna()

# Separate features & target
X_tabular = df.drop(['median_house_value'], axis=1)
y = df['median_house_value'] #output is median_house_value

# Convert categorical columns (like ocean_proximity) into numeric format
# Using One-Hot Encoding
X_tabular = pd.get_dummies(X_tabular)

# Scale/normalize tabular features
# StandardScaler transforms features to have mean=0 and std=1
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_tabular_scaled = scaler.fit_transform(X_tabular)

In [ ]:
# Upload zipped image dataset
uploaded = files.upload()

# Unzip image files into current directory
!unzip images.zip

Saving 996.jpg to 996.jpg
Saving 997.jpg to 997.jpg
Saving 998.jpg to 998.jpg
Saving 999.jpg to 999.jpg
Saving 9952.jpg to 9952.jpg
Saving 9953.jpg to 9953.jpg
Saving 9954.jpg to 9954.jpg
Saving 9955.jpg to 9955.jpg
Saving 9956.jpg to 9956.jpg
Saving 9957.jpg to 9957.jpg
Saving 9958.jpg to 9958.jpg
Saving 9959.jpg to 9959.jpg
Saving 9960.jpg to 9960.jpg
Saving 9961.jpg to 9961.jpg
Saving 9962.jpg to 9962.jpg
Saving 9963.jpg to 9963.jpg
Saving 9964.jpg to 9964.jpg
Saving 9965.jpg to 9965.jpg
Saving 9966.jpg to 9966.jpg
Saving 9967.jpg to 9967.jpg
Saving 9968.jpg to 9968.jpg
Saving 9969.jpg to 9969.jpg
Saving 9970.jpg to 9970.jpg
Saving 9971.jpg to 9971.jpg
Saving 9972.jpg to 9972.jpg
Saving 9973.jpg to 9973.jpg
Saving 9974.jpg to 9974.jpg
Saving 9975.jpg to 9975.jpg
Saving 9976.jpg to 9976.jpg
Saving 9977.jpg to 9977.jpg
Saving 9978.jpg to 9978.jpg
Saving 9979.jpg to 9979.jpg
Saving 9980.jpg to 9980.jpg
Saving 9981.jpg to 9981.jpg
Saving 9982.jpg to 9982.jpg
Saving 9983.jpg to 9983.jpg


In [ ]:
import os

# Get list of all .jpg image files from directory
image_files = [f for f in os.listdir("/content") if f.endswith(".jpg")]

# Sort images to maintain consistent order
image_files = sorted(image_files)

# Rename images sequentially (0.jpg, 1.jpg, 2.jpg, ...)
# This ensures image index matches dataset row index
for i, filename in enumerate(image_files):
    old_path = os.path.join("/content", filename)
    new_path = os.path.join("/content", f"{i}.jpg")
    os.rename(old_path, new_path)

print("Renaming done!")

Renaming done!


In [ ]:
# Ensure dataset size matches number of images
# Keep only as many rows as available images
df = df.iloc[:len(image_files)]

# Add image_id column to link each row with its corresponding image
df['image_id'] = df.index

In [ ]:
# Import required libraries for image processing
import cv2
import numpy as np
import os
from tensorflow.keras.applications.resnet50 import preprocess_input

# Define image size required by ResNet50
IMAGE_SIZE = 224

# Function to load and preprocess images
def load_images(image_ids, folder_path):
    images = []

    for img_id in image_ids:
       # Construct image path (0.jpg, 1.jpg, etc.)
        img_path = os.path.join(folder_path, f"{img_id}.jpg")

        if os.path.exists(img_path):
            # Read image
            img = cv2.imread(img_path)
            # Resize to 224x224 (required by ResNet50)
            img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
            # Apply ResNet50 preprocessing
            img = preprocess_input(img)
             # If image missing, append black image
            images.append(img)
        else:
            images.append(np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3)))

    return np.array(images)


In [ ]:
# Load images corresponding to dataset rows
X_images = load_images(df['image_id'], "/content")

In [ ]:
print("Number of images:", X_images.shape[0])

Number of images: 485


In [ ]:
N = X_images.shape[0]


# Ensure tabular data and target match image count
X_tabular_scaled = X_tabular_scaled[:N]
y = y[:N]

print("Tabular shape:", X_tabular_scaled.shape)
print("Images shape:", X_images.shape)
print("Target shape:", y.shape)

Tabular shape: (485, 13)
Images shape: (485, 224, 224, 3)
Target shape: (485,)


In [ ]:
from sklearn.model_selection import train_test_split

#### You split:

# 80% for training

# 20% for testing

# And you split:

# Tabular data

# Image data

# Target values

X_tab_train, X_tab_test, X_img_train, X_img_test, y_train, y_test = train_test_split(
    X_tabular_scaled,
    X_images,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50

# Load pretrained ResNet50 model (without top classification layer)
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False   # freeze weights

image_input = layers.Input(shape=(224,224,3))


#Image → ResNet50 → Feature maps→ GlobalAveragePooling → Convert to vector→ Dense(128) → Learned image representation
x = base_model(image_input)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)

image_branch = models.Model(inputs=image_input, outputs=x)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:

#Tabular features → Dense(128) → Dense(64)
tabular_input = layers.Input(shape=(X_tab_train.shape[1],))

y_tab = layers.Dense(128, activation='relu')(tabular_input)
y_tab = layers.Dense(64, activation='relu')(y_tab)

tabular_branch = models.Model(inputs=tabular_input, outputs=y_tab)

In [ ]:
#Final Regression Layer

combined = layers.concatenate([image_branch.output, tabular_branch.output])

z = layers.Dense(128, activation='relu')(combined)
z = layers.Dense(1)(z)   # regression output

model = models.Model(
    inputs=[image_branch.input, tabular_branch.input],
    outputs=z
)
#This creates the complete multimodal architecture.

In [ ]:
#Compile Model
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer_1[0]… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 13)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │      1,792 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    262,272 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     24,704 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 1)         │        129 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,884,865 (91.11 MB)

 Trainable params: 297,153 (1.13 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
#Train Model
history = model.fit(
    [X_img_train, X_tab_train],
    y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=32
)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 78s 6s/step - loss: 45372616704.0000 - mae: 190753.5000 - val_loss: 36044267520.0000 - val_mae: 172639.6094
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 62s 6s/step - loss: 44455649280.0000 - mae: 190000.2656 - val_loss: 35972083712.0000 - val_mae: 172429.0469
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 67s 6s/step - loss: 44845391872.0000 - mae: 189483.3594 - val_loss: 35821010944.0000 - val_mae: 171987.5156
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 81s 6s/step - loss: 44000944128.0000 - mae: 188709.9688 - val_loss: 35546001408.0000 - val_mae: 171180.8281
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 65s 6s/step - loss: 44860108800.0000 - mae: 188526.3438 - val_loss: 35089416192.0000 - val_mae: 169832.7812
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 82s 6s/step - loss: 43323346944.0000 - mae: 185834.5000 - val_loss: 34382426112.0000 - val_mae: 167723.2188
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 78s 6s/step - loss: 44919246848.0000 - mae: 189274.9844 - val_loss: 33347999744.0000 -

In [ ]:
#Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

predictions = model.predict([X_img_test, X_tab_test])

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("MAE:", mae)
print("RMSE:", rmse)

4/4 ━━━━━━━━━━━━━━━━━━━━ 21s 4s/step
MAE: 156545.99341575385
RMSE: 186712.1405604743


In [ ]:
# For California housing dataset:
# Prices range around:
# 200k – 500k

#So:
# Error ~150k is quite high.

#Possible reasons:

# Only 10 epochs

# Frozen ResNet (no fine-tuning)

# Image dataset may not strongly correlate with price

# Small dataset size